установка необходимых библиотек

In [20]:
! pip install datasets docx2txt langchain langchain_community

In [1]:
import openai
import numpy as np
from typing import List

from langchain_community.document_loaders import Docx2txtLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

from google.colab import drive
drive.mount('/content/drive')

## Инициализация клиента и моделей

In [33]:
client = openai.OpenAI(
    api_key="you_token",
    base_url="https://api.vsellm.ru/v1"
)

embed_model_name = 'openai/text-embedding-3-small'
generative_model_name = "openai/gpt-4.1-mini"

## Пример использования генеративных моделей

Генерация эмбедингов

In [31]:
texts = [
    "Graph Neural Networks are powerful",
    "GNNs are used in recommender systems",
    "Transformers work well for NLP"
]

response = client.embeddings.create(
    model=embed_model_name,
    input=texts
)

embeddings = [item.embedding for item in response.data]

print(len(embeddings))          # количество текстов
print(len(embeddings[0]))       # размерность эмбеддинга (3072)


3
1536


Генерация текста

In [32]:
response = client.chat.completions.create(
    model=generative_model_name,
    messages=[
        {"role": "user", "content": "Привет!"}
    ]
)

print(response.choices[0].message.content)


Привет! Как я могу помочь?


## Baseline HACK2025


## Загрузка базы знаний

In [21]:
loader = Docx2txtLoader(
    "/content/drive/My Drive/ИТМО/Course_GNN_materials/Materials/Курс.docx"
)

documents = loader.load()

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)

chunks = splitter.split_documents(documents)

print(f"Total chunks: {len(chunks)}")



Total chunks: 100


## Генерация эмбеддингов для чанков

In [24]:
def embed_documents(docs):
    texts = [doc.page_content for doc in docs]
    response = client.embeddings.create(
        model=embed_model_name,
        input=texts
    )
    return np.array([item.embedding for item in response.data])


chunk_embeddings = embed_documents(chunks)



## Retrieval (cosine similarity)

In [25]:
def cosine_sim(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))


def retrieve(query, k=5):
    query_emb = client.embeddings.create(
        model=embed_model_name,
        input=[query]
    ).data[0].embedding

    scores = [
        cosine_sim(query_emb, emb)
        for emb in chunk_embeddings
    ]

    top_idx = np.argsort(scores)[-k:][::-1]

    return [
        {
            "text": chunks[i].page_content,
            "metadata": chunks[i].metadata,
            "score": float(scores[i])
        }
        for i in top_idx
    ]


## Сбор контекста

In [27]:
def build_context(retrieved_chunks):
    return "\n\n".join(
        f"[score={c['score']:.3f}]\n{c['text']}"
        for c in retrieved_chunks
    )


## Генерация ответа

In [28]:
def generate_answer(question, context):
    response = client.chat.completions.create(
        model=generative_model_name,
        messages=[
            {
                "role": "system",
                "content": (
                    "Ты ассистент, отвечающий на вопросы по курсу Graph Neural Networks. "
                    "Используй только предоставленный контекст."
                )
            },
            {
                "role": "user",
                "content": f"""
Контекст:
{context}

Вопрос:
{question}

Дай краткий и точный ответ.
"""
            }
        ],
        temperature=0.2
    )

    return response.choices[0].message.content


## Финальная функция answer()

In [29]:
def answer(question, k=5):
    retrieved = retrieve(question, k)
    context = build_context(retrieved)
    answer_text = generate_answer(question, context)

    return {
        "question": question,
        "answer": answer_text,
        "retrieved_chunks": retrieved
    }


In [30]:
result = answer("Что такое Graph Neural Network?")

print(result["answer"])


Graph Neural Network (GNN) — это специализированный класс нейросетей, разработанный для работы с графами, который учитывает сложные взаимосвязи и зависимости между объектами, эффективно моделируя структуру и топологию графовых данных.


#### Где можно вдохновиться: RAG From Scratch